# Step 3 — Compute Calibration Statistics for the Simulator

This notebook completes **Step 3 only**.

Goal: learn transition, duration, branch, and arrival statistics from real case-step data, then save calibration artifacts for Step 6.

## What we do in this step (simple view)

1. Load `case_step_features` from Step 2.
2. Compute transition probabilities (municipality + global).
3. Compute duration statistics per activity.
4. Compute branch probabilities at decision points.
5. Estimate case-arrival proxy from first-event timestamps.
6. Save `transition_stats.csv`, `duration_stats.csv`, and `sim_calibration.json`.

In [1]:
%pip install numpy


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

DATASET_DIR = Path('./dataset')
OUTPUT_DIR = Path('./output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURES_PARQUET = OUTPUT_DIR / 'case_step_features.parquet'
FEATURES_CSV = OUTPUT_DIR / 'case_step_features.csv'
TRANSITION_STATS_PATH = OUTPUT_DIR / 'transition_stats.csv'
DURATION_STATS_PATH = OUTPUT_DIR / 'duration_stats.csv'
SIM_CALIBRATION_PATH = OUTPUT_DIR / 'sim_calibration.json'

print('Dataset dir       :', DATASET_DIR.resolve())
print('Output dir        :', OUTPUT_DIR.resolve())
print('Expected features :', FEATURES_PARQUET.resolve())

Dataset dir       : /home/praharsha/Desktop/bueracratic-workflow-analyzer/dataset
Output dir        : /home/praharsha/Desktop/bueracratic-workflow-analyzer/output
Expected features : /home/praharsha/Desktop/bueracratic-workflow-analyzer/output/case_step_features.parquet


In [3]:
if not FEATURES_PARQUET.exists():
    print('Error: previous notebook should be run first (Step 2 missing case_step_features.parquet).')
    raise SystemExit(1)

features_df = pd.read_parquet(FEATURES_PARQUET)
loaded_from = FEATURES_PARQUET

features_df['timestamp'] = pd.to_datetime(features_df['timestamp'], utc=True, errors='coerce')
features_df = features_df.sort_values(['municipality', 'case_id', 'timestamp', 'event_id']).reset_index(drop=True)

print(f'Loaded {len(features_df):,} rows from {loaded_from.name}')
print('Columns:', len(features_df.columns))
features_df[['municipality', 'case_id', 'activity', 'next_activity', 'time_since_prev_hours']].head(5)

Loaded 262,628 rows from case_step_features.parquet
Columns: 33


,municipality,case_id,activity,next_activity,time_since_prev_hours
0,1,44688,register submission date request,phase application received,0.000000
1,1,44688,phase application received,terminate on request,248.995278
2,1,44688,terminate on request,applicant is stakeholder,0.001944
3,1,44688,applicant is stakeholder,no permit needed or only notification needed,0.001389
4,1,44688,no permit needed or only notification needed,send confirmation receipt,0.001944


## 3.1 Transition probabilities

We estimate `P(next_activity | activity)` per municipality and globally.

In [4]:
transition_base = features_df.dropna(subset=['activity', 'next_activity']).copy()

# Counts the activity -> next_activity transition occurance in municipality
transition_counts_m = (
    transition_base
    .groupby(['municipality', 'activity', 'next_activity'], dropna=False)
    .size()
    .reset_index(name='transition_count')
)

# Counts the total number of outgoing transitions from activity in municipality
totals_m = (
    transition_counts_m
    .groupby(['municipality', 'activity'], dropna=False)['transition_count']
    .sum()
    .reset_index(name='source_total')
)

# Calculate transition probabilities for municipality-level transitions
transition_stats_m = transition_counts_m.merge(totals_m, on=['municipality', 'activity'], how='left')
transition_stats_m['transition_prob'] = transition_stats_m['transition_count'] / transition_stats_m['source_total']

# Repeat the same for global transitions (ignoring municipality)
transition_counts_g = (
    transition_base
    .groupby(['activity', 'next_activity'], dropna=False)
    .size()
    .reset_index(name='transition_count')
)

totals_g = (
    transition_counts_g
    .groupby(['activity'], dropna=False)['transition_count']
    .sum()
    .reset_index(name='source_total')
)

transition_stats_g = transition_counts_g.merge(totals_g, on=['activity'], how='left')
transition_stats_g['transition_prob'] = transition_stats_g['transition_count'] / transition_stats_g['source_total']
transition_stats_g['municipality'] = 'global'

transition_stats = pd.concat([transition_stats_m, transition_stats_g], ignore_index=True)
transition_stats = transition_stats[['municipality', 'activity', 'next_activity', 'transition_count', 'source_total', 'transition_prob']]
transition_stats = transition_stats.sort_values(['municipality', 'activity', 'transition_prob'], ascending=[True, True, False]).reset_index(drop=True)

transition_stats.to_csv(TRANSITION_STATS_PATH, index=False)
print('Saved:', TRANSITION_STATS_PATH.resolve())
transition_stats.head(10)

Saved: /home/praharsha/Desktop/bueracratic-workflow-analyzer/output/transition_stats.csv


,municipality,activity,next_activity,transition_count,source_total,transition_prob
0,1,MER present,activities regular procedure,92,93,0.989247
1,1,MER present,inform BAG administrator,1,93,0.010753
2,1,MER present in supplement,request complete,86,171,0.502924
3,1,MER present in supplement,WAW permit aspect,33,171,0.192982
4,1,MER present in supplement,treat subcases completeness,17,171,0.099415
5,1,MER present in supplement,phase additional information received,8,171,0.046784
6,1,MER present in supplement,receive additional information,4,171,0.023392
7,1,MER present in supplement,extend procedure term,3,171,0.017544
8,1,MER present in supplement,creating environmental permit decision,2,171,0.011696
9,1,MER present in supplement,procedure change,2,171,0.011696


In [5]:
# Check: transition rows should sum to ~1 per (municipality, activity)
row_sums = (
    transition_stats
    .groupby(['municipality', 'activity'])['transition_prob']
    .sum()
    .reset_index(name='prob_sum')
)

# Calculate absolute error from 1.0 for each row and find max error and rows outside tolerance
row_sums['abs_error_from_1'] = (row_sums['prob_sum'] - 1.0).abs()
max_abs_error = row_sums['abs_error_from_1'].max()
bad_rows = row_sums[row_sums['abs_error_from_1'] > 1e-9]

print('Max |sum(prob)-1|:', float(max_abs_error))
print('Rows outside tolerance (1e-9):', len(bad_rows))
row_sums.sort_values('abs_error_from_1', ascending=False).head(10)

Max |sum(prob)-1|: 1.1102230246251565e-16
Rows outside tolerance (1e-9): 0


,municipality,activity,prob_sum,abs_error_from_1
816,3,set decision phase suspension decided,1.0,1.110223e-16
1664,global,register number of days extension period,1.0,1.110223e-16
1717,global,set phase,1.0,1.110223e-16
1019,4,phase asked additional information,1.0,1.110223e-16
158,1,phase decision irrevocable,1.0,1.110223e-16
1061,4,register number of days extension period,1.0,1.110223e-16
1710,global,set decision phase suspension sent,1.0,1.110223e-16
1182,5,create letter requesting missing data,1.0,0.000000e+00
1183,5,create letter requesting updated plan,1.0,0.000000e+00
1184,5,create message terminate request,1.0,0.000000e+00


## 3.2 Duration and branch statistics

Durations are based on `time_since_prev_hours` (excluding first step in each case).

Branch probabilities use robust labels from Step 2:
- prefer `branch_label` with confidence filtering,
- fallback to legacy boolean flags only if robust labels are unavailable.

Decision points are activities with at least two distinct observed next activities.

In [6]:
duration_base = features_df[features_df['step_index'] > 0].copy()
duration_base = duration_base[duration_base['time_since_prev_hours'].notna()].copy()
duration_base = duration_base[duration_base['time_since_prev_hours'] >= 0].copy()

# For each municipality and activity, calculate count, mean, median, std, min, 25% quantile, 75% quantile, max of time_since_prev_hours
# These will be used for calibrating the duration distribution of the simulator's activity durations, and for sanity checking the time_since_prev_hours values in the features dataset
def build_duration_stats(df: pd.DataFrame, municipality_value):
    grouped = df.groupby('activity')['time_since_prev_hours']
    out = grouped.agg(
        obs_count='count',
        duration_mean_hours='mean',
        duration_median_hours='median',
        duration_std_hours='std',
        duration_min_hours='min',
        duration_max_hours='max'
    ).reset_index()

    q25 = grouped.quantile(0.25).reset_index(name='duration_q25_hours')
    q75 = grouped.quantile(0.75).reset_index(name='duration_q75_hours')

    log_mean = (
        df[df['time_since_prev_hours'] > 0]
        .assign(_log_duration=np.log(df.loc[df['time_since_prev_hours'] > 0, 'time_since_prev_hours']))
        .groupby('activity')['_log_duration']
        .mean()
        .reset_index(name='duration_log_mean')
    )

    out = out.merge(q25, on='activity', how='left').merge(q75, on='activity', how='left').merge(log_mean, on='activity', how='left')
    out['municipality'] = municipality_value
    return out

duration_stats_m = []
for m, g in duration_base.groupby('municipality'):
    duration_stats_m.append(build_duration_stats(g, int(m)))

duration_stats_m = pd.concat(duration_stats_m, ignore_index=True)
duration_stats_g = build_duration_stats(duration_base, 'global')
duration_stats = pd.concat([duration_stats_m, duration_stats_g], ignore_index=True)
duration_stats = duration_stats[[
    'municipality', 'activity', 'obs_count',
    'duration_mean_hours', 'duration_median_hours', 'duration_std_hours',
    'duration_min_hours', 'duration_q25_hours', 'duration_q75_hours', 'duration_max_hours',
    'duration_log_mean'
]]
duration_stats = duration_stats.sort_values(['municipality', 'obs_count'], ascending=[True, False]).reset_index(drop=True)

duration_stats.to_csv(DURATION_STATS_PATH, index=False)
print('Saved:', DURATION_STATS_PATH.resolve())
duration_stats.head(10)

Saved: /home/praharsha/Desktop/bueracratic-workflow-analyzer/output/duration_stats.csv


,municipality,activity,obs_count,duration_mean_hours,duration_median_hours,duration_std_hours,duration_min_hours,duration_q25_hours,duration_q75_hours,duration_max_hours,duration_log_mean
0,1,send confirmation receipt,2074,21.307012,0.000278,345.930242,0.0,0.000000,0.003889,12426.746389,-4.781218
1,1,procedure change,1752,39.485848,0.000556,368.987736,0.0,0.000000,0.001111,12336.000000,-5.838542
2,1,enter senddate decision environmental permit,1397,29.465748,0.000000,242.967571,0.0,0.000000,11.224167,8485.180556,0.852134
3,1,request complete,1267,87.590212,0.000833,464.297208,0.0,0.000000,0.002222,8775.115556,-4.497491
4,1,phase application received,1199,30.592633,0.000000,97.947934,0.0,0.000000,11.105000,1142.858889,2.440490
5,1,forward to the competent authority,1177,16.583966,0.001111,184.724622,0.0,0.000000,0.002222,5687.201667,-4.985685
6,1,regular procedure without MER,1161,39.212005,0.000000,380.140996,0.0,0.000000,0.001389,7081.096389,-5.363483
7,1,enter senddate acknowledgement,1051,42.648477,8.912500,111.956855,0.0,0.000278,15.043889,1848.000000,1.219897
8,1,phase application receptive,940,18.439295,0.000000,565.337070,0.0,0.000000,0.000000,17332.906389,-7.805720
9,1,article 34 WABO applies,933,8.871008,0.000278,105.569425,0.0,0.000000,0.000833,1602.706389,-6.961236


In [7]:
# Decision-point activities: source activities with >=2 distinct next activities
decision_points = (
    features_df.dropna(subset=['next_activity'])
    .groupby(['municipality', 'activity'])['next_activity']
    .nunique()
    .reset_index(name='num_next_activities')
)
decision_points = decision_points[decision_points['num_next_activities'] >= 2]

branch_base = features_df.merge(
    decision_points[['municipality', 'activity']],
    on=['municipality', 'activity'],
    how='inner'
)

# Robust branch estimation path (preferred)
BRANCH_CONFIDENCE_MIN = 0.70
uses_robust_labels = {'branch_label', 'branch_confidence'}.issubset(branch_base.columns)

if uses_robust_labels:
    branch_base_for_probs = branch_base[
        (branch_base['branch_label'].notna())
        & (branch_base['branch_label'] != 'unknown')
        & (branch_base['branch_confidence'] >= BRANCH_CONFIDENCE_MIN)
    ].copy()

    branch_probs = (
        branch_base_for_probs.groupby(['municipality', 'activity'], dropna=False)
        .agg(
            n_obs=('activity', 'size'),
            refusal_prob=('branch_label', lambda s: float((s == 'refusal').mean())),
            suspension_prob=('branch_label', lambda s: float((s == 'suspension').mean())),
            completeness_prob=('branch_label', lambda s: float((s == 'completeness').mean())),
            mean_branch_confidence=('branch_confidence', 'mean')
        )
        .reset_index()
        .sort_values(['municipality', 'n_obs'], ascending=[True, False])
        .reset_index(drop=True)
    )

    branch_method = f'robust_label_confidence_filtered>= {BRANCH_CONFIDENCE_MIN}'
else:
    # Backward-compatible fallback
    branch_base_for_probs = branch_base.copy()
    branch_probs = (
        branch_base_for_probs.groupby(['municipality', 'activity'], dropna=False)
        .agg(
            n_obs=('activity', 'size'),
            refusal_prob=('is_refusal_path', 'mean'),
            suspension_prob=('is_suspension_path', 'mean'),
            completeness_prob=('is_completeness_path', 'mean')
        )
        .reset_index()
        .sort_values(['municipality', 'n_obs'], ascending=[True, False])
        .reset_index(drop=True)
    )
    branch_probs['mean_branch_confidence'] = np.nan
    branch_method = 'legacy_boolean_flags'

first_events = (
    features_df.sort_values(['municipality', 'case_id', 'timestamp'])
    .groupby(['municipality', 'case_id'], as_index=False)
    .first()[['municipality', 'case_id', 'timestamp']]
)

arrival_rows = []
for m, g in first_events.groupby('municipality'):
    g = g.sort_values('timestamp').reset_index(drop=True)
    delta_h = g['timestamp'].diff().dt.total_seconds() / 3600.0
    delta_h = delta_h[delta_h.notna() & (delta_h >= 0)]

    if len(delta_h) == 0:
        arrival_rows.append({
            'municipality': int(m),
            'case_count': int(len(g)),
            'interarrival_count': 0,
            'interarrival_mean_hours': None,
            'interarrival_median_hours': None,
            'interarrival_q25_hours': None,
            'interarrival_q75_hours': None
        })
    else:
        arrival_rows.append({
            'municipality': int(m),
            'case_count': int(len(g)),
            'interarrival_count': int(len(delta_h)),
            'interarrival_mean_hours': float(delta_h.mean()),
            'interarrival_median_hours': float(delta_h.median()),
            'interarrival_q25_hours': float(delta_h.quantile(0.25)),
            'interarrival_q75_hours': float(delta_h.quantile(0.75))
        })

arrival_proxy = pd.DataFrame(arrival_rows).sort_values('municipality').reset_index(drop=True)

sim_calibration = {
    'notes': 'Generated by step3_calibration_stats.ipynb',
    'transition_stats_path': str(TRANSITION_STATS_PATH),
    'duration_stats_path': str(DURATION_STATS_PATH),
    'branch_probability_method': branch_method,
    'branch_confidence_min': BRANCH_CONFIDENCE_MIN if uses_robust_labels else None,
    'branch_probabilities': {
        str(int(m)): [
            {
                'activity': row['activity'],
                'n_obs': int(row['n_obs']),
                'refusal_prob': float(row['refusal_prob']),
                'suspension_prob': float(row['suspension_prob']),
                'completeness_prob': float(row['completeness_prob']),
                'mean_branch_confidence': None if pd.isna(row['mean_branch_confidence']) else float(row['mean_branch_confidence'])
            }
            for _, row in grp.iterrows()
        ]
        for m, grp in branch_probs.groupby('municipality')
    },
    'arrival_proxy_by_municipality': [
        {
            'municipality': int(row['municipality']),
            'case_count': int(row['case_count']),
            'interarrival_count': int(row['interarrival_count']),
            'interarrival_mean_hours': None if pd.isna(row['interarrival_mean_hours']) else float(row['interarrival_mean_hours']),
            'interarrival_median_hours': None if pd.isna(row['interarrival_median_hours']) else float(row['interarrival_median_hours']),
            'interarrival_q25_hours': None if pd.isna(row['interarrival_q25_hours']) else float(row['interarrival_q25_hours']),
            'interarrival_q75_hours': None if pd.isna(row['interarrival_q75_hours']) else float(row['interarrival_q75_hours'])
        }
        for _, row in arrival_proxy.iterrows()
    ]
}

with open(SIM_CALIBRATION_PATH, 'w') as f:
    json.dump(sim_calibration, f, indent=2)

print('Saved:', SIM_CALIBRATION_PATH.resolve())
print('Branch method:', branch_method)
print('Decision-point rows:', len(branch_probs))
if uses_robust_labels:
    print('Rows kept for branch probabilities:', len(branch_base_for_probs), 'out of', len(branch_base))
arrival_proxy

Saved: /home/praharsha/Desktop/bueracratic-workflow-analyzer/output/sim_calibration.json
Branch method: robust_label_confidence_filtered>= 0.7
Decision-point rows: 1047
Rows kept for branch probabilities: 142202 out of 262042


,municipality,case_count,interarrival_count,interarrival_mean_hours,interarrival_median_hours,interarrival_q25_hours,interarrival_q75_hours
0,1,1199,1198,32.153589,24.0,0.0,48.0
1,2,832,831,48.981949,24.0,0.0,72.0
2,3,1409,1408,32.147727,24.0,0.0,48.0
3,4,1053,1052,44.053232,24.0,0.0,48.0
4,5,1156,1155,40.000000,24.0,0.0,48.0


In [8]:
# Validation checks for Step 3 acceptance

# 1) Transition rows sum to ~1
sum_check = (
    transition_stats
    .groupby(['municipality', 'activity'])['transition_prob']
    .sum()
    .reset_index(name='prob_sum')
)
sum_check_ok = bool((sum_check['prob_sum'] - 1.0).abs().max() < 1e-9)

# 2) No empty duration model for frequent activities
MIN_FREQ = 50
freq_activities = (
    duration_base.groupby(['municipality', 'activity']).size().reset_index(name='n')
    .query('n >= @MIN_FREQ')
)
dur_coverage = freq_activities.merge(
    duration_stats[duration_stats['municipality'] != 'global'][['municipality', 'activity', 'obs_count']],
    on=['municipality', 'activity'],
    how='left'
)
missing_duration_models = dur_coverage[dur_coverage['obs_count'].isna()]

print('Transition probability sum check passed:', sum_check_ok)
print(f'Frequent activity threshold: n >= {MIN_FREQ}')
print('Frequent activities:', len(freq_activities))
print('Missing duration models for frequent activities:', len(missing_duration_models))

if len(missing_duration_models) > 0:
    display(missing_duration_models.head(20))

Transition probability sum check passed: True
Frequent activity threshold: n >= 50
Frequent activities: 572
Missing duration models for frequent activities: 0


## Step 3 complete

You now have simulator calibration artifacts in `./output`:
- `transition_stats.csv`
- `duration_stats.csv`
- `sim_calibration.json`

Next step: build and validate the calibrated simulator (Steps 6–7).